**Дисциплина:** «Технологии обработки больших данных»  
**Группа:** ИД24-3  
**ФИО:** Бугарева Елизавета Ивановна 


**Тема:** «Исследование туристических данных для выявления сезонных и географических закономерностей спроса»  
 
## Туристический спрос на примере двух гостиниц Португалии
## 2.1–2.3

# 2.1. Постановка задачи
**Задание:** определить цель, критерии достижения, границы, объект, предмет и гипотезу.

**Тема:** подготовка данных для сравнения сезонности бронирований двух гостиниц Португалии.

**Бизнес-проблема:** число бронирований и отмен меняется по месяцам. Эти изменения нужно учитывать при планировании работы гостиницы.

**Бизнес-цель:** определить месяцы высокого и низкого числа бронирований в каждом из двух отелей. Пока я только готовлю данные для этого.

**Критерии:** получить полную таблицу из 24 строк (2 отеля × 12 месяцев), без пропусков в числе бронирований и погодных показателях, с уникальным ключом «отель — месяц». На следующих этапах сравнить месяцы и проверить гипотезу.

**Объект:** бронирования одного городского отеля в Лиссабоне и одного курортного отеля в Алгарве.

**Предмет:** изменение числа бронирований и отмен по месяцу планируемого заезда; погода рассматривается как дополнительная характеристика месяца.

**Границы:** только 2016 год и эти два отеля. Исходная строка — одна бронь, итоговая строка — один отель за месяц. Число броней не равно числу туристов или ночёвок. По этим данным нельзя описать всю Португалию или все городские и курортные отели.

**Гипотеза:** распределение бронирований по месяцам различается между двумя отелями; у курортного отеля доля летних бронирований выше, чем у городского.

**Пояснение:** это предположение, которое будет проверяться в 2.4. В 2015 и 2017 годах набор охватывает не все месяцы, поэтому для простого сравнения беру полный 2016 год. Тексты нужны для описания направлений и выполнения обработки неструктурированных данных, а не для доказательства гипотезы.

# 2.2. Источники 
**Задание:** подобрать табличные данные, XML, API JSON и текст; показать структуру и выбрать признаки.

| Формат | Источник | Что находится внутри |
|---|---|---|
| CSV → реляционная таблица SQLite | Hotel Booking Demand, копия TidyTuesday | Одна строка — бронь, столбцы — её характеристики |
| JSON API | Open-Meteo Historical Weather, ERA5 | Даты и массивы суточной погоды |
| XML | Экспорт Wikivoyage: Lisbon и Algarve | Статьи, редакции, даты и текст с разметкой |
| Неструктурированный текст | Тексты этих статей Wikivoyage | Описания направлений обычным языком |

 CSV — табличная выгрузка из гостиничных систем, которую ниже сохраняю в SQLite.


In [1]:
from pathlib import Path
import json, io, zipfile, base64, hashlib, sqlite3
import xml.etree.ElementTree as ET
import pandas as pd
import mwparserfromhell
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display

snapshot = None
for file in sorted(Path('.').glob('*.ipynb'), key=lambda p: p.stat().st_mtime, reverse=True):
    snapshot = json.loads(file.read_text(encoding='utf-8')).get('metadata', {}).get('tourism_sources_v2')
    if snapshot:
        break
assert snapshot, 'Положите этот ноутбук в рабочую папку Jupyter.'
data_dir = Path('tourism_data')
with zipfile.ZipFile(io.BytesIO(base64.b64decode(snapshot))) as archive:
    archive.extractall(data_dir)
manifest = json.loads((data_dir / 'manifest.json').read_text())
for item in manifest:
    assert hashlib.sha256((data_dir / item['file']).read_bytes()).hexdigest() == item['sha256']
pd.set_option('display.max_colwidth', 90)
display(pd.DataFrame(manifest)[['file', 'status', 'bytes', 'checked']])

,file,status,bytes,checked
0,hotels.csv,200,16855599,2026-11-02
1,weather_lisbon.json,200,10628,2026-11-02
2,weather_algarve.json,200,10630,2026-11-02
3,wikivoyage.xml,200,118268,2026-11-02


### Источник 1. Hotel Booking Demand — бронирования
**Авторы:** Nuno Antonio, Ana de Almeida, Luis Nunes. [Научная статья и описание получения данных](https://doi.org/10.1016/j.dib.2018.11.126), [копия статьи в репозитории университета](https://repositorio.iscte-iul.pt/bitstream/10071/16929/1/Hotelbookingdemanddatasets.pdf).

Авторы описывают выгрузку из реальных гостиничных систем PMS и удаление сведений, позволяющих идентифицировать отель или клиента. Период планируемых заездов: 01.07.2015–31.08.2017. Есть состоявшиеся и отменённые бронирования.

**Откуда скачан файл:** [CSV TidyTuesday](https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv). Это открытая копия, а не сайт гостиницы. [Описание TidyTuesday](https://github.com/rfordatascience/tidytuesday/blob/main/data/2020/2020-02-11/readme.md) прямо ссылается на статью и содержит словарь полей.

Получено 119 390 строк и 32 столбца: 40 060 Resort Hotel и 79 330 City Hotel — объёмы совпадают с публикацией. В статье у каждого отеля 31 признак; в общем CSV добавлен столбец `hotel`.

**Что покажет код:** размер таблицы, первые пять строк выбранных полей и число строк по отелям. Полный исходный CSV остаётся в папке `tourism_data`.

| Поле | Значение |
|---|---|
| `hotel` | City Hotel — городской отель; Resort Hotel — курортный |
| `is_canceled` | 1 — бронь отменена, 0 — не отменена |
| `arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month` | Год, месяц и день планируемого заезда |
| `lead_time` | Число дней между внесением брони и заездом |
| `stays_in_week_nights`, `stays_in_weekend_nights` | Забронированные ночи по будням и выходным |
| `adr` | Средняя дневная стоимость по определению авторов; денежный показатель, не суммарная выручка |
| `market_segment`, `customer_type` | Сегмент продажи и тип клиента |

Далее использую только эти поля. Полное описание остальных столбцов находится в словаре TidyTuesday. Открытая статья опубликована по CC BY 4.0; авторство и ссылка сохранены.

In [2]:
raw = pd.read_csv(data_dir / 'hotels.csv')
columns = ['hotel', 'is_canceled', 'arrival_date_year', 'arrival_date_month',
           'arrival_date_day_of_month', 'lead_time', 'stays_in_week_nights',
           'stays_in_weekend_nights', 'adr', 'market_segment', 'customer_type']
print('Размер исходного CSV:', raw.shape)
display(raw[columns].head())
display(raw['hotel'].value_counts().to_frame('Бронирования'))
assert raw.shape == (119390, 32)
assert raw['hotel'].value_counts().to_dict() == {'City Hotel': 79330, 'Resort Hotel': 40060}

Размер исходного CSV: (119390, 32)


,hotel,is_canceled,arrival_date_year,arrival_date_month,arrival_date_day_of_month,lead_time,stays_in_week_nights,stays_in_weekend_nights,adr,market_segment,customer_type
0,Resort Hotel,0,2015,July,1,342,0,0,0.0,Direct,Transient
1,Resort Hotel,0,2015,July,1,737,0,0,0.0,Direct,Transient
2,Resort Hotel,0,2015,July,1,7,1,0,75.0,Direct,Transient
3,Resort Hotel,0,2015,July,1,13,1,0,75.0,Corporate,Transient
4,Resort Hotel,0,2015,July,1,14,2,0,98.0,Online TA,Transient


,Бронирования
hotel,
City Hotel,79330
Resort Hotel,40060


### Источник 2. Open-Meteo — погода в JSON
[Документация Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api). Запрос передаёт координаты, начало и конец периода, список показателей и `models=era5`. В ответе `daily` содержит массивы дат и значений, а `daily_units` — единицы измерения.

Это **метеорологический реанализ ERA5**, сочетающий наблюдения и модель атмосферы.Атрибуция: Weather data by Open-Meteo.com, CC BY 4.0.

Беру 366 дней 2016 года. Для Лиссабона использую точку 38.72, −9.14; для Алгарве — Фару, 37.02, −7.93. Фару служит приближённым представителем региона: точный адрес курортного отеля неизвестен. 

| Поле JSON | Смысл |
|---|---|
| `time` | Дата |
| `temperature_2m_mean` | Средняя суточная температура на высоте 2 м, °C |
| `precipitation_sum` | Осадки за сутки, мм |
| `wind_speed_10m_max` | Максимальная за сутки скорость ветра на высоте 10 м, км/ч |

Ниже показаны ключи, единицы и первые строки. Полные URL запросов сохранены в таблице `manifest` и выводятся в конце ноутбука.

In [3]:
locations = {'City Hotel': 'Lisbon', 'Resort Hotel': 'Algarve'}
parts = []
for place, name in [('Lisbon', 'weather_lisbon.json'), ('Algarve', 'weather_algarve.json')]:
    answer = json.loads((data_dir / name).read_text())
    parts.append(pd.DataFrame(answer['daily']).assign(location=place))
weather_raw = pd.concat(parts, ignore_index=True)
print('Ключи ответа:', list(answer))
print('Единицы:', answer['daily_units'])
print('Размер:', weather_raw.shape)
display(weather_raw.head())

Ключи ответа: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Единицы: {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}
Размер: (732, 5)


,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,location
0,2016-01-01,14.7,7.7,35.6,Lisbon
1,2016-01-02,13.8,0.0,15.1,Lisbon
2,2016-01-03,15.6,2.0,29.4,Lisbon
3,2016-01-04,15.7,7.5,39.8,Lisbon
4,2016-01-05,11.7,2.5,39.7,Lisbon


### Источник 3. Wikivoyage — XML и текст
Использую англоязычные статьи [Lisbon](https://en.wikivoyage.org/wiki/Lisbon) и [Algarve](https://en.wikivoyage.org/wiki/Algarve). [MediaWiki API](https://www.mediawiki.org/wiki/API:Query) возвращает экспорт XML по параметрам `action=query`, `titles=Lisbon|Algarve`, `export=1`, `exportnowrap=1`.

**Структура XML:** `mediawiki → page → revision → text`. У страницы есть название, у редакции — номер, дата и текст с викиразметкой. `exportnowrap=1` возвращает сам XML даже при служебном параметре `format=json`.

**Что беру:** название, номер редакции, время редакции, текст, число символов, разделов и внутренних ссылок. Последние три — характеристики документа(не количество достопримечательностей). В следующем разделе очищаю этот же текст, считаю слова и назначаю простую тематическую метку.

**Ограничение:** это современные статьи, а не архивные описания 2016 года и не отзывы туристов. Они используются как справочный контекст. По ним не измеряется удовлетворённость и не объясняется исторический спрос.

Авторы — участники Wikivoyage. Тексты CC BY-SA 4.0; ссылки на конкретные редакции сохраняются. Очистка разметки — выполненное здесь преобразование. [Условия Wikimedia](https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

In [4]:
xml = (data_dir / 'wikivoyage.xml').read_text()
print('Начало XML:')
print(xml[:450])
root = ET.fromstring(xml)
ns = {'m': root.tag.split('}')[0][1:]}
rows = []
for page in root.findall('m:page', ns):
    revision = page.find('m:revision', ns)
    text = revision.findtext('m:text', default='', namespaces=ns)
    revision_id = revision.findtext('m:id', namespaces=ns)
    rows.append({'location': page.findtext('m:title', namespaces=ns),
                 'revision_id': revision_id,
                 'revision_date': revision.findtext('m:timestamp', namespaces=ns),
                 'source': 'https://en.wikivoyage.org/w/index.php?oldid=' + revision_id,
                 'raw_text': text, 'char_count': len(text),
                 'section_count': len(mwparserfromhell.parse(text).filter_headings()),
                 'link_count': len(mwparserfromhell.parse(text).filter_wikilinks())})
wiki = pd.DataFrame(rows)
wiki['hotel'] = wiki['location'].map({v: k for k, v in locations.items()})
display(wiki.drop(columns='raw_text').head())
print('Пример исходного текста:')
print(wiki.iloc[0]['raw_text'][:600])

Начало XML:
<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.11/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.11/ http://www.mediawiki.org/xml/export-0.11.xsd" version="0.11" xml:lang="en">
  <siteinfo>
    <sitename>Wikivoyage</sitename>
    <dbname>enwikivoyage</dbname>
    <base>https://en.wikivoyage.org/wiki/Main_Page</base>
    <generator>MediaWiki 1.48.0-wmf.1</generator>
    <c
Пример исходного текста:
{{pagebanner|Praia da Marinha Banner.JPG}}

The '''Algarve''' is the southernmost region of [[Portugal]], on the coast of the '''Atlantic Ocean'''. It is Portugal's most popular holiday destination due to the approximately {{km|200}} of clean beaches, the cool, unpolluted water, and the fact that it is relatively cheap, very safe, and overall welcoming.

==Cities==
{{mapframe|width=550|height=300|staticmap=Algarve Map.png}}

The cities of the Algarve are traditionally grouped into '''Barlavento''' (Windward

,location,revision_id,revision_date,source,char_count,section_count,link_count,hotel
0,Algarve,5252981,2026-04-03T17:06:34Z,https://en.wikivoyage.org/w/index.php?oldid=5252981,20725,25,86,Resort Hotel
1,Lisbon,5409287,2026-10-29T19:52:57Z,https://en.wikivoyage.org/w/index.php?oldid=5409287,93499,64,211,City Hotel


### Какие признаки оставляю

| Набор | Категориальные признаки | Числовые признаки | Роль |
|---|---|---|---|
| Бронирования | `hotel`, `market_segment`, `customer_type` | `lead_time`, `nights`, `adr` | Описание брони; количество строк и отмен — показатели спроса |
| Погода | `location`, `month`, `season` | `temp_c`, `precip_mm`, `wind_kmh` | Характеристики месяца, дополнительные факторы |
| XML | `location`, `hotel`, `theme` | `char_count`, `section_count`, `link_count` | Происхождение и структура статей |
| Текст | `location`, `hotel`, `theme` | `word_count`, `culture_count`, `nature_count` | Простое описание содержания |

Текстов всего два, их достаточно для показа обработки, но мало для самостоятельного статистического исследования.
 `is_canceled` — бинарная категория.

**Вывод 2.2:** все выбранные файлы получены и прочитаны. Ни один источник не заменяется пустой таблицей 

# 2.3. Подготовка данных
**Задание:** проверить пропуски, выполнить простую очистку, аннотирование и векторизацию текста, слияние и сохранение.

### 1. Проверка таблицы бронирований
Оставляю 2016 год и нужные столбцы. Показываю типы, пропуски и количество одинаковых строк полного CSV за этот год. Дубликаты **не удаляю**: идентификатора брони нет, и две настоящие брони могут иметь одинаковые признаки.

In [5]:
bookings = raw.loc[raw['arrival_date_year'] == 2016, columns].copy()
print('Бронирований за 2016 год:', len(bookings))
display(pd.DataFrame({'Тип': bookings.dtypes.astype(str), 'Пропуски': bookings.isna().sum()}))
print('Полностью одинаковых строк:', raw.loc[raw['arrival_date_year'] == 2016].duplicated().sum())

Бронирований за 2016 год: 56707
Полностью одинаковых строк: 14316


,Тип,Пропуски
hotel,object,0
is_canceled,int64,0
arrival_date_year,int64,0
arrival_date_month,object,0
arrival_date_day_of_month,int64,0
lead_time,int64,0
stays_in_week_nights,int64,0
stays_in_weekend_nights,int64,0
adr,float64,0
market_segment,object,0


### 2. Минимальная очистка бронирований
Убираю лишние пробелы в категориях, собираю дату и суммирую будничные и выходные ночи. Пустую категорию обозначаю `Unknown`. Невозможные отрицательные значения длительности и цены заменяю пропусками. Нулевую цену и нулевую длительность автоматически не удаляю.

Для подсчёта броней необходимы отель, дата и признак отмены. Проверяю их: если ключ испорчен, выполнение остановится с ошибкой вместо незаметного удаления брони. Пропуски в дополнительных показателях видны в выводе.

In [6]:
for column in ['hotel', 'market_segment', 'customer_type']:
    bookings[column] = bookings[column].str.strip().replace('', pd.NA).fillna('Unknown')
months = {name: i + 1 for i, name in enumerate([
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'])}
bookings['date'] = pd.to_datetime(dict(year=bookings['arrival_date_year'],
    month=bookings['arrival_date_month'].map(months),
    day=bookings['arrival_date_day_of_month']), errors='coerce')
bookings['nights'] = bookings['stays_in_week_nights'] + bookings['stays_in_weekend_nights']
for column in ['lead_time', 'nights', 'adr']:
    bookings[column] = pd.to_numeric(bookings[column], errors='coerce')
    bookings.loc[bookings[column] < 0, column] = float('nan')
bookings['location'] = bookings['hotel'].map(locations)
bookings['month'] = bookings['date'].dt.month
assert bookings[['date', 'location', 'is_canceled']].notna().all().all()
assert bookings['is_canceled'].isin([0, 1]).all()
display(bookings[['hotel', 'date', 'is_canceled', 'nights', 'adr']].head())
display(bookings[['lead_time', 'nights', 'adr']].isna().sum().to_frame('Пропуски после очистки'))

,hotel,date,is_canceled,nights,adr
3738,Resort Hotel,2016-01-01,0,1.0,59.94
3739,Resort Hotel,2016-01-01,0,1.0,116.10
3740,Resort Hotel,2016-01-01,1,1.0,89.00
3741,Resort Hotel,2016-01-01,0,2.0,73.46
3742,Resort Hotel,2016-01-01,1,2.0,119.00


,Пропуски после очистки
lead_time,0
nights,0
adr,0


### 3. Погода: от дней к месяцам
Сначала показываю пропуски и повторные ключи «территория — дата». Преобразую даты, удаляю повторы дней и рассчитываю среднюю температуру, сумму осадков и среднее суточных максимумов ветра.

Перед агрегацией проверяю наличие всех 366 дней для каждой территории. Не заполняю отсутствующую погоду средним или нулём.

In [7]:
display(weather_raw.isna().sum().to_frame('Пропуски погоды'))
print('Повторные дни:', weather_raw.duplicated(['location', 'time']).sum())
daily = weather_raw.drop_duplicates(['location', 'time']).copy()
daily['date'] = pd.to_datetime(daily['time'])
assert daily.notna().all().all()
expected = set(pd.date_range('2016-01-01', '2016-12-31'))
assert all(set(group['date']) == expected for _, group in daily.groupby('location'))
daily['month'] = daily['date'].dt.month
weather = daily.groupby(['location', 'month'], as_index=False).agg(
    temp_c=('temperature_2m_mean', 'mean'),
    precip_mm=('precipitation_sum', 'sum'),
    wind_kmh=('wind_speed_10m_max', 'mean'))
seasons = {1:'Зима', 2:'Зима', 3:'Весна', 4:'Весна', 5:'Весна', 6:'Лето',
           7:'Лето', 8:'Лето', 9:'Осень', 10:'Осень', 11:'Осень', 12:'Зима'}
weather['season'] = weather['month'].map(seasons)
display(weather.head())

Повторные дни: 0


,Пропуски погоды
time,0
temperature_2m_mean,0
precipitation_sum,0
wind_speed_10m_max,0
location,0


,location,month,temp_c,precip_mm,wind_kmh,season
0,Algarve,1,14.525806,35.7,27.606452,Зима
1,Algarve,2,13.403448,13.1,33.572414,Зима
2,Algarve,3,12.874194,5.1,27.090323,Весна
3,Algarve,4,14.926667,29.7,26.656667,Весна
4,Algarve,5,17.077419,104.1,29.032258,Весна


### 4. Очистка текста и простая аннотация
Удаляю викиразметку и лишние пробелы. Считаю слова и совпадения с двумя маленькими английскими словарями: культура и природа. Больше совпадений с первым — метка «культура», со вторым — «природа»; равенство — «смешанная», ни одного совпадения — «не определена».

Это учебная автоматическая разметка, она может ошибаться. Слова в статье не измеряют реальное количество объектов. Показываю текст после очистки, чтобы преобразование было видно.

In [8]:
texts = wiki.copy()
texts['text'] = texts['raw_text'].apply(lambda s: mwparserfromhell.parse(s).strip_code())
texts['text'] = texts['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
assert texts['text'].str.len().gt(0).all() and texts['location'].is_unique
texts['word_count'] = texts['text'].str.split().str.len()
lower = texts['text'].str.lower()
texts['culture_count'] = lower.str.count(r'\b(?:museums?|palaces?|churches|church|theatres?|theaters?)\b')
texts['nature_count'] = lower.str.count(r'\b(?:beaches|beach|mountains?|forests?|lakes?)\b')
texts['theme'] = 'смешанная'
texts.loc[texts['culture_count'] > texts['nature_count'], 'theme'] = 'культура'
texts.loc[texts['nature_count'] > texts['culture_count'], 'theme'] = 'природа'
texts.loc[texts['culture_count'] + texts['nature_count'] == 0, 'theme'] = 'не определена'
display(texts[['location', 'hotel', 'word_count', 'culture_count', 'nature_count', 'theme']])
print(texts.iloc[0]['text'][:700])

The Algarve is the southernmost region of Portugal, on the coast of the Atlantic Ocean. It is Portugal's most popular holiday destination due to the approximately of clean beaches, the cool, unpolluted water, and the fact that it is relatively cheap, very safe, and overall welcoming. Cities The cities of the Algarve are traditionally grouped into Barlavento (Windward) and Sotavento (Leeward), the western and eastern halves, respectively. Barlavento — the inland Algarve, superb food, nature and views — close to famous surf spots — town and municipality containing other destinations: — near Europe's southwesternmost point — narrow, winding streets, quaint shops and wonderful restaurants, with 


,location,hotel,word_count,culture_count,nature_count,theme
0,Algarve,Resort Hotel,2535,2,14,природа
1,Lisbon,City Hotel,10268,18,4,культура


### 5. Векторизация TF-IDF
Перевожу два текста в небольшую числовую таблицу: строка — статья, столбец — слово, значение — его вес. Убираю распространённые английские служебные слова и оставляю только 10 признаков. Это демонстрация векторизации, без обучения модели.

In [9]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=10)
vectors = vectorizer.fit_transform(texts['text'])
tfidf = pd.DataFrame(vectors.toarray(), index=texts['location'],
                     columns=vectorizer.get_feature_names_out())
display(tfidf.round(3))

,airport,algarve,bus,card,city,lisbon,metro,portugal,portuguese,thumb
location,,,,,,,,,,
Algarve,0.184,0.897,0.207,0.092,0.161,0.184,0.000,0.161,0.138,0.069
Lisbon,0.141,0.006,0.135,0.154,0.301,0.817,0.294,0.197,0.160,0.172


### 6. Объединение и хранение
Брони группирую по отелю, территории и месяцу **планируемого заезда**. `bookings` — все брони, `cancellations` — отменённые, `not_canceled` — неотменённые. Последний показатель не называю числом туристов или подтверждённых проживаний. Дата отмены здесь не используется.

Погоду присоединяю по `location + month`, статьи — по `location`. Год один (2016), поэтому месяц однозначен. При расширении на другие годы в ключ нужно добавить год. Проверка `validate` не позволяет незаметно размножить строки при соединении.

Для учебного объёма выбираю **SQLite**: один файл, реляционные таблицы и доступ через SQL. Он подходит для этих данных; распределённое хранилище здесь не требуется. Сохраняю исходные выбранные брони, погоду, статьи, TF-IDF и итоговую месячную таблицу. Для просмотра итог также записываю в CSV.

In [10]:
def merge_and_save():
    monthly = bookings.groupby(['hotel', 'location', 'month'], as_index=False).agg(
        bookings=('is_canceled', 'size'), cancellations=('is_canceled', 'sum'))
    monthly['not_canceled'] = monthly['bookings'] - monthly['cancellations']
    result = monthly.merge(weather, on=['location', 'month'], validate='one_to_one')
    features = texts[['location', 'theme', 'char_count', 'section_count', 'link_count',
                      'word_count', 'culture_count', 'nature_count']]
    result = result.merge(features, on='location', validate='many_to_one')
    assert len(result) == 24 and not result.duplicated(['hotel', 'month']).any()
    assert result.notna().all().all()
    assert result['bookings'].sum() == len(bookings)
    result.to_csv(data_dir / 'prepared_data.csv', index=False)
    with sqlite3.connect(data_dir / 'tourism.sqlite') as connection:
        for name, table in [('bookings', bookings), ('weather', weather),
                            ('articles', texts), ('text_tfidf', tfidf.reset_index()),
                            ('prepared_data', result)]:
            table.to_sql(name, connection, if_exists='replace', index=False)
    return result

result = merge_and_save()
print('Размер итоговой таблицы:', result.shape)
display(result.head(12).round(2))
display(result.isna().sum().to_frame('Пропуски'))

Размер итоговой таблицы: (24, 17)


,hotel,location,month,bookings,cancellations,not_canceled,temp_c,precip_mm,wind_kmh,season,theme,char_count,section_count,link_count,word_count,culture_count,nature_count
0,City Hotel,Lisbon,1,1364,438,926,13.63,76.4,24.77,Зима,культура,93499,64,211,10268,18,4
1,City Hotel,Lisbon,2,2371,930,1441,12.38,65.3,29.87,Зима,культура,93499,64,211,10268,18,4
2,City Hotel,Lisbon,3,3046,1108,1938,12.20,25.6,25.01,Весна,культура,93499,64,211,10268,18,4
3,City Hotel,Lisbon,4,3561,1539,2022,13.90,68.7,26.90,Весна,культура,93499,64,211,10268,18,4
4,City Hotel,Lisbon,5,3676,1436,2240,15.92,130.1,24.56,Весна,культура,93499,64,211,10268,18,4
5,City Hotel,Lisbon,6,3923,1720,2203,19.20,2.3,24.72,Лето,культура,93499,64,211,10268,18,4
6,City Hotel,Lisbon,7,3131,1043,2088,21.20,3.0,25.22,Лето,культура,93499,64,211,10268,18,4
7,City Hotel,Lisbon,8,3378,1247,2131,21.64,0.0,25.51,Лето,культура,93499,64,211,10268,18,4
8,City Hotel,Lisbon,9,3871,1567,2304,20.47,15.7,22.46,Осень,культура,93499,64,211,10268,18,4
9,City Hotel,Lisbon,10,4219,1947,2272,18.08,70.6,19.51,Осень,культура,93499,64,211,10268,18,4


,Пропуски
hotel,0
location,0
month,0
bookings,0
cancellations,0
not_canceled,0
temp_c,0
precip_mm,0
wind_kmh,0
season,0


### 7. Повторное скачивание источников
Чтобы обновить данные, надо поставить `UPDATE = True`, выполнить эту ячейку, затем заново ячейки от чтения CSV до `merge_and_save()`. Ячейку распаковки при этом пропустите: она восстанавливает сохранённый снимок. Запросы погоды останутся за 2016 год, а статьи могут получить новые редакции. При ошибке загрузки код остановится; подставных данных не будет. Повторная загрузка не изменяет встроенную копию ноутбука.

In [11]:
UPDATE = False
if UPDATE:
    import requests
    downloaded = {}
    for item in manifest:
        response = requests.get(item['url'], timeout=90,
            headers={'User-Agent': 'TourismCoursework/1.0 (educational project)'})
        response.raise_for_status()
        downloaded[item['file']] = response.content
    for name, content in downloaded.items():
        (data_dir / name).write_bytes(content)
    print('Источники скачаны. Повторите чтение и обработку, пропустив распаковку.')
else:
    print('Используется проверенный встроенный снимок, сетевых запросов нет.')

Используется проверенный встроенный снимок, сетевых запросов нет.


### Итог по пунктам 2.1–2.3
Сформулирована задача, описаны источники, показаны исходные таблицы и выполнена минимальная подготовка. Для 2016 года используются **56 707 реальных записей бронирований**, **732 дневных погодных наблюдения реанализа** и **2 статьи**. Получены **24 месячные строки без пропусков**. Данные сохранены в CSV и SQLite.

Пока Современные тексты остаются справочным контекстом, а погодная связь сама по себе не доказывает причину бронирования.

Проверяемое происхождение:

Ниже находятся точные URL и SHA-256 исходных файлов. Хеш подтверждает неизменность встроенной копии относительно скачанного файла, но сам по себе не доказывает истинность наблюдений. Доступность сайтов проверена 02.11.2026, постоянная доступность не предполагается — поэтому исходники вложены в ноутбук.

In [12]:
for item in manifest:
    print(item['file'])
    print('URL:', item['url'])
    print('SHA-256:', item['sha256'])
    print()

hotels.csv
URL: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv
SHA-256: 7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06

weather_lisbon.json
URL: https://archive-api.open-meteo.com/v1/archive?latitude=38.72&longitude=-9.14&start_date=2016-01-01&end_date=2016-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Cwind_speed_10m_max&timezone=Europe%2FLisbon&models=era5
SHA-256: c035b906cfa5432e98c1ce35307351644dbc293ddc36065d34caac1073623808

weather_algarve.json
URL: https://archive-api.open-meteo.com/v1/archive?latitude=37.02&longitude=-7.93&start_date=2016-01-01&end_date=2016-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Cwind_speed_10m_max&timezone=Europe%2FLisbon&models=era5
SHA-256: 2099d13b1c56c0a58326888f873ceeedfb69a6ef18defb7c59f6212d2a03ac5e

wikivoyage.xml
URL: https://en.wikivoyage.org/w/api.php?action=query&titles=Lisbon%7CAlgarve&export=1&exportnowrap=1&format=json
SHA-256: 5932242d1e40d5a061fd747